In [1]:
import os
import torch
import numpy as np
import pandas as pd
from epiweeks import Week
import preprocess_data as prep
import model as mc
import matplotlib.pyplot as plt 

pd.options.mode.chained_assignment = None

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

regioes_estados = {
        'Sul': ['SC', 'PR', 'RS'],
        'Sudeste': ['SP', 'MG', 'RJ', 'ES'],
        'Nordeste': ['BA', 'CE', 'PE', 'PB', 'PI', 'RN', 'MA', 'AL', 'SE'],
        'Centro-Oeste': ['DF', 'MT', 'MS', 'GO'],
        'Norte': ['RO', 'AC', 'AM', 'RR', 'PA', 'AP', 'TO']
    } 
    
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

columns_to_normalize = ['casos','epiweek', 'biome', 'enso']

predict_n = 30
max_epiweek = 22
    
boxcox = False
min_year = 2015

TEST_YEAR = 2026
media = False

In [2]:
doenca = 'dengue'
model_name = 'mix_22_26_V3'
base_model = 'base_22_26'
enso_model = 'enso_22_26_V3'
filename = f'./data/{doenca}.csv.gz'

df = prep.load_cases_data(filename)

df = df.loc[df.epiweek <= int(f'{TEST_YEAR}{max_epiweek}')]

enso = prep.load_enso_weekly(filename='data/enso_weekly_forecast_up_25_07.csv')

enso_neutro = prep.load_enso_weekly(filename='data/enso_weekly_neutro_up_25_07.csv')


In [3]:
def agg_sum_samples(df_samples):
    """
    Converte o DataFrame plano de amostras do Parquet em tabelas de 
    intervalos de confiança (quantis) para cada regional e data.
    
    Parameters
    ----------
    df_samples : pd.DataFrame
        DataFrame lido do parquet contendo as colunas: 
        ['regional_geocode', 'pass_id', 'date', 'predicted_cases']
    """
    
    # 1. Agrupamos por regional e data para calcular os percentis sobre a coluna de casos
    # O groupby garante que o cálculo do quantil rode isolado para cada par (regional, data)
    grouped = df_samples.groupby(['date', 'pass_id'])['predicted_cases'].sum().reset_index().groupby('date')['predicted_cases']

    # 2. Construímos o DataFrame de intervalos calculando os quantis do Pandas
    df_preds = pd.DataFrame({
        'pred': grouped.quantile(0.50),

        'lower_50': grouped.quantile(0.25),
        'upper_50': grouped.quantile(0.75),

        'lower_80': grouped.quantile(0.10),
        'upper_80': grouped.quantile(0.90),

        'lower_90': grouped.quantile(0.05),
        'upper_90': grouped.quantile(0.95),

        'lower_95': grouped.quantile(0.025),
        'upper_95': grouped.quantile(0.975),
    }).reset_index() # Retorna 'regional_geocode' e 'date' de índices para colunas normais

    # 3. Garante a formatação correta da data
    df_preds['date'] = pd.to_datetime(df_preds['date'])
    
    # Ordena para ficar visualmente fácil de analisar (Regional -> Linha do tempo)
    df_preds = df_preds.sort_values(by=['date']).reset_index(drop=True)

    return df_preds


def agg_total_cases(df_samples):
    """
    Converte o DataFrame plano de amostras do Parquet em tabelas de 
    intervalos de confiança (quantis) para cada regional e data.
    
    Parameters
    ----------
    df_samples : pd.DataFrame
        DataFrame lido do parquet contendo as colunas: 
        ['regional_geocode', 'pass_id', 'date', 'predicted_cases']
    """
    
    # 1. Agrupamos por regional e data para calcular os percentis sobre a coluna de casos
    # O groupby garante que o cálculo do quantil rode isolado para cada par (regional, data)
    grouped = df_samples.groupby(['pass_id'])['predicted_cases'].sum().reset_index()

    # 2. Construímos o DataFrame de intervalos calculando os quantis do Pandas
    df_preds = pd.DataFrame({
        'pred': [np.percentile(grouped['predicted_cases'], q=50) ],

        'lower_50':[ np.percentile(grouped['predicted_cases'], q=25)],
        'upper_50': [np.percentile(grouped['predicted_cases'], q=75)],

        'lower_80': [np.percentile(grouped['predicted_cases'], q=10)],
        'upper_80': [np.percentile(grouped['predicted_cases'], q=90)],

        'lower_90': [np.percentile(grouped['predicted_cases'], q=5)],
        'upper_90': [np.percentile(grouped['predicted_cases'], q=95)],

        'lower_95': [np.percentile(grouped['predicted_cases'], q=2.5)],
        'upper_95': [np.percentile(grouped['predicted_cases'], q=97.5)],
    })
    
    return df_preds

Apply_model:

El nino: 

In [4]:
df_agg_total_state = pd.DataFrame()
df_agg_total_region = pd.DataFrame()

for region in regioes_estados.keys(): 

    print(region)

    label = f'{region}_{TEST_YEAR-1}_{model_name}'


    df_preds_region = pd.DataFrame()
    
    for state in regioes_estados[region]: 

        df_reg = df.loc[df.uf ==state]
        df_reg = df_reg.loc[df_reg.index >= pd.to_datetime(Week(2015,1).startdate())]
        

        model = mc.load_model(
                            region,
                            TEST_YEAR,
                            doenca,
                            model_name,
                            predict_n,
                            max_epiweek,
                            device,
                            base_model,
                            enso_model
                        )  
            
        df_preds =  mc.save_regional_samples_to_parquet(
                                                            model,
                                                            df_reg,
                                                            enso,
                                                            TEST_YEAR,
                                                            columns_to_normalize,
                                                            max_epiweek=max_epiweek,
                                                            boxcox=boxcox,
                                                            n_passes=500,
                                                            min_year=min_year,
                                                            media=True)
        
        df_preds_region = pd.concat([df_preds_region, df_preds], ignore_index=True)

        df_agg = agg_sum_samples(df_preds)
        df_agg.to_csv( f'predictions/preds_{doenca}_{state}_{TEST_YEAR}_{model_name}.csv', index=False)


        df_agg_total = agg_total_cases(df_preds)
        df_agg_total['state'] = state
        df_agg_total['model_name'] = 'El niño'
        df_agg_total_state = pd.concat([df_agg_total_state, df_agg_total], ignore_index = True)

    df_preds_region.to_parquet(f'predictions/samples__{doenca}_{region}_{TEST_YEAR}_{model_name}.parquet')
    df_agg_region = agg_sum_samples(df_preds_region)
    df_agg_region.to_csv( f'predictions/preds_{doenca}_{region}_{TEST_YEAR}_{model_name}.csv', index=False)
    
    df_agg_total_reg = agg_total_cases(df_preds_region)
    df_agg_total_reg['region'] = region
    df_agg_total_reg['model_name'] = 'El niño'
    df_agg_total_region = pd.concat([df_agg_total_region, df_agg_total_reg], ignore_index = True)


df_agg_total_region.to_csv( f'predictions/preds_total_region_{doenca}_{TEST_YEAR}_{model_name}.csv',index=False)
df_agg_total_state.to_csv(f'predictions/preds_total_state_{doenca}_{TEST_YEAR}_{model_name}.csv', index=False)


Sul
Sudeste
Nordeste
Centro-Oeste
Norte


NeutrO: 

In [ ]:
df_agg_total_state = pd.DataFrame()
df_agg_total_region = pd.DataFrame()

for region in regioes_estados.keys(): 

    print(region)

    label = f'{region}_{TEST_YEAR-1}_{model_name}'


    df_preds_region = pd.DataFrame()
    
    for state in regioes_estados[region]: 

        df_reg = df.loc[df.uf ==state]
        df_reg = df_reg.loc[df_reg.index >= pd.to_datetime(Week(2015,1).startdate())]
        

        model = mc.load_model(
                            region,
                            TEST_YEAR,
                            doenca,
                            model_name,
                            predict_n,
                            max_epiweek,
                            device,
                            base_model,
                            enso_model
                        )  
            
        df_preds =  mc.save_regional_samples_to_parquet(
                                                            model,
                                                            df_reg,
                                                            enso_neutro,
                                                            TEST_YEAR,
                                                            columns_to_normalize,
                                                            max_epiweek=max_epiweek,
                                                            boxcox=boxcox,
                                                            n_passes=500,
                                                            min_year=min_year,
                                                            media=True)
        
        df_preds_region = pd.concat([df_preds_region, df_preds], ignore_index=True)

        df_agg = agg_sum_samples(df_preds)
        df_agg.to_csv( f'predictions/preds_neutro_{doenca}_{state}_{TEST_YEAR}_{model_name}.csv', index=False)


        df_agg_total = agg_total_cases(df_preds)
        df_agg_total['state'] = state
        df_agg_total['model_name'] = 'Neutro'
        df_agg_total_state = pd.concat([df_agg_total_state, df_agg_total], ignore_index = True)

    df_preds_region.to_parquet(f'predictions/samples_neutro_{doenca}_{region}_{TEST_YEAR}_{model_name}.parquet')
    df_agg_region = agg_sum_samples(df_preds_region)
    df_agg_region.to_csv( f'predictions/preds_neutro_{doenca}_{region}_{TEST_YEAR}_{model_name}.csv', index=False)
    
    df_agg_total_reg = agg_total_cases(df_preds_region)
    df_agg_total_reg['region'] = region
    df_agg_total_reg['model_name'] = 'Neutro'
    df_agg_total_region = pd.concat([df_agg_total_region, df_agg_total_reg], ignore_index = True)


df_agg_total_region.to_csv( f'predictions/preds_neutro_total_region_{doenca}_{TEST_YEAR}_{model_name}.csv',index=False)
df_agg_total_state.to_csv(f'predictions/preds_neutro_total_state_{doenca}_{TEST_YEAR}_{model_name}.csv', index=False)


Sul
Sudeste
Nordeste
Centro-Oeste
Norte
